# 📰 HFC Weekly Financial Intelligence Newsletter
**HFC Group Plc — Automated News Aggregation System**

This notebook scrapes, filters, summarises, and packages a weekly newsletter
from 8 financial news sources. Run all cells top-to-bottom to generate the newsletter.

---

## 1️⃣  Install Dependencies

In [ ]:
# Run once per Colab session
!pip install -q requests beautifulsoup4 newspaper3k sumy lxml[html_clean] pandas
print('✅ All dependencies installed.')

## 2️⃣  Configuration

In [ ]:
# ── User-configurable settings ────────────────────────────────────────────
DAYS_BACK   = 7       # How many days back to look for articles
OUTPUT_DIR  = '/content/hfc_output'   # Where to save output files
MAX_ARTICLES_PER_SOURCE = 10          # Cap per source (saves time in Colab)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

## 3️⃣  Load Module Code

In [ ]:
# === SCRAPER ===
import requests
from bs4 import BeautifulSoup
import logging, time, random, re
from datetime import datetime, timedelta
from urllib.parse import urlparse
from dataclasses import dataclass, field
from typing import Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('hfc')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

SOURCES = {
    'Central Bank of Kenya': {'url':'https://www.centralbank.go.ke/press-releases/','article_selector':'h3.entry-title a, h2.entry-title a, .post-title a','date_selector':'.entry-date, .post-date, time'},
    'Business Daily Africa': {'url':'https://www.businessdailyafrica.com/bd/markets','article_selector':'h3 a, h2 a, .article-title a','date_selector':'.date, time'},
    'Kenyan Wall Street':    {'url':'https://kenyanwallstreet.com/category/banking/','article_selector':'h2.entry-title a, h3.entry-title a','date_selector':'time.entry-date, .published'},
    'Nation Africa':         {'url':'https://nation.africa/kenya/business','article_selector':'h3 a, h2 a, .story-title a','date_selector':'time, .date-published'},
    'Reuters Finance':       {'url':'https://www.reuters.com/business/finance/','article_selector':'a[data-testid="Heading"], h3 a','date_selector':'time'},
    'Business Today Kenya':  {'url':'https://businesstoday.co.ke/','article_selector':'h3 a, h2 a, .entry-title a','date_selector':'.entry-date, time'},
}

@dataclass
class Article:
    title: str
    url: str
    source: str
    pub_date: Optional[datetime] = None
    body: str = ''
    summary: str = ''
    takeaway: str = ''
    category: str = ''

def get_with_retry(url, retries=3):
    for attempt in range(1, retries+1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r
        except Exception as e:
            logger.warning(f'Attempt {attempt}/{retries} failed: {e}')
            time.sleep(2**attempt)
    return None

def parse_date(text):
    for fmt in ['%B %d, %Y','%b %d, %Y','%Y-%m-%d','%d %B %Y','%Y-%m-%dT%H:%M:%S']:
        try: return datetime.strptime(text.strip()[:20], fmt)
        except: pass
    return None

print('✅ Scraper definitions loaded.')

In [ ]:
# === ARTICLE PARSER ===
try:
    from newspaper import Article as Np3kArticle
    NP_OK = True
except: NP_OK = False

def parse_article(url):
    if NP_OK:
        try:
            a = Np3kArticle(url); a.download(); a.parse()
            if len(a.text) > 150: return a.text.strip()
        except: pass
    r = get_with_retry(url)
    if not r: return ''
    soup = BeautifulSoup(r.text, 'html.parser')
    for t in soup(['script','style','nav','header','footer','aside']): t.decompose()
    for sel in ['article','[itemprop="articleBody"]','.article-body','.post-content','.entry-content','main']:
        el = soup.select_one(sel)
        if el:
            txt = el.get_text(' ', strip=True)
            if len(txt) > 150: return txt
    return ''

print('✅ Article parser loaded.')

In [ ]:
# === CLEANER ===
from difflib import SequenceMatcher

FINANCE_KW = ['bank','banking','mortgage','interest rate','loan','inflation','credit',
              'central bank','fintech','housing finance','regulation','investment',
              'bond','equity','market','economy','monetary','treasury','forex','gdp',
              'sacco','nse','cbk','kenya shilling']

def has_finance_kw(art):
    txt = f'{art.title} {art.body}'.lower()
    return any(kw in txt for kw in FINANCE_KW)

def norm(text): return re.sub(r'\W+',' ', text.lower()).strip()

def similar(a, b, t=0.85): return SequenceMatcher(None,a,b).ratio() >= t

def clean_pipeline(articles):
    cutoff = datetime.now() - timedelta(days=DAYS_BACK)
    # Date filter
    arts = [a for a in articles if a.pub_date is None or a.pub_date >= cutoff]
    logger.info(f'After date filter: {len(arts)}')
    # Keyword filter
    arts = [a for a in arts if has_finance_kw(a)]
    logger.info(f'After keyword filter: {len(arts)}')
    # Dedup
    seen_u, seen_t, unique = set(), [], []
    for a in arts:
        if a.url in seen_u: continue
        nt = norm(a.title)
        if any(similar(nt, t) for t in seen_t): continue
        seen_u.add(a.url); seen_t.append(nt); unique.append(a)
    logger.info(f'After dedup: {len(unique)}')
    # Clean body text
    for a in unique:
        a.body = re.sub(r'[^\x20-\x7E\n]', ' ', a.body)
        a.body = re.sub(r'\s+', ' ', a.body).strip()
    return unique

print('✅ Cleaner loaded.')

In [ ]:
# === SUMMARISER & CATEGORISER ===
try:
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.lsa import LsaSummarizer
    from sumy.nlp.stemmers import Stemmer
    from sumy.utils import get_stop_words
    SUMY_OK = True
except: SUMY_OK = False

def summarise(text, n=3):
    if not text or len(text.split()) < 30: return text[:300]
    if SUMY_OK:
        try:
            p = PlaintextParser.from_string(text, Tokenizer('english'))
            s = LsaSummarizer(Stemmer('english'))
            s.stop_words = get_stop_words('english')
            return ' '.join(str(x) for x in s(p.document, n))
        except: pass
    sents = re.split(r'(?<=[.!?])\s+', text)
    return ' '.join(sents[:n])

def takeaway(text, title=''):
    for p in [r'(?:announced?|said|reported?|confirmed?)[^.]{10,80}\.',
               r'(?:will|plans? to|expects? to)[^.]{10,80}\.',
               r'(?:increased?|decreased?|rose|fell)[^.]{5,60}\.' ]:
        m = re.search(p, text, re.IGNORECASE)
        if m and 20 < len(m.group(0)) < 200: return m.group(0).strip()
    for s in re.split(r'(?<=[.!?])\s+', text):
        if len(s) > 40: return s[:200]
    return title

CATS = {
    'Kenyan Banking News':       ['kenya','kenyan','kcb','equity bank','hfc','housing finance','absa','ncba','cbk'],
    'Economic Updates':          ['gdp','inflation','economy','economic','growth','budget','fiscal','monetary'],
    'Regulatory & Policy Changes':['regulation','policy','law','compliance','directive','approval','license'],
    'Fintech & Technology':      ['fintech','mobile money','mpesa','digital','blockchain','crypto','payment','app'],
    'Global Banking Trends':     ['federal reserve','fed','global','international','world bank','wall street'],
}

def categorise(art):
    txt = f'{art.title} {art.body}'.lower()
    scores = {c: sum(1 for k in kws if k in txt) for c, kws in CATS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'General Finance'

print('✅ Summariser & categoriser loaded.')

## 4️⃣  Run the Pipeline

In [ ]:
# ── STEP 1: Scrape ────────────────────────────────────────────────────────
all_articles = []
seen_urls = set()
cutoff = datetime.now() - timedelta(days=DAYS_BACK)

for source_name, cfg in SOURCES.items():
    print(f'\n🔍 Scraping: {source_name}')
    resp = get_with_retry(cfg['url'])
    if not resp:
        print(f'  ⚠️  Failed to reach {cfg["url"]}')
        continue
    soup = BeautifulSoup(resp.text, 'html.parser')
    links = soup.select(cfg['article_selector'])
    count = 0
    for tag in links[:MAX_ARTICLES_PER_SOURCE * 3]:
        if count >= MAX_ARTICLES_PER_SOURCE: break
        href = tag.get('href',''); title = tag.get_text(strip=True)
        if not href or len(title) < 10: continue
        if href.startswith('/'):
            base = urlparse(cfg['url']); href = f'{base.scheme}://{base.netloc}{href}'
        elif not href.startswith('http'): continue
        if href in seen_urls: continue
        pub_date = None
        parent = tag.parent
        for _ in range(5):
            if not parent: break
            de = parent.select_one(cfg['date_selector'])
            if de: pub_date = parse_date(de.get('datetime') or de.get_text()); break
            parent = parent.parent
        if pub_date and pub_date < cutoff: continue
        all_articles.append(Article(title=title, url=href, source=source_name, pub_date=pub_date))
        seen_urls.add(href); count += 1
        time.sleep(random.uniform(0.3, 0.7))
    print(f'  ✅ {count} articles collected')
    time.sleep(random.uniform(1.5, 2.5))

print(f'\n📦 Total collected: {len(all_articles)} articles')

In [ ]:
# ── STEP 2: Parse article bodies ─────────────────────────────────────────
print(f'Parsing body text for {len(all_articles)} articles…\n')
for i, art in enumerate(all_articles):
    print(f'  [{i+1}/{len(all_articles)}] {art.title[:60]}…')
    art.body = parse_article(art.url)
    time.sleep(random.uniform(0.8, 1.5))
print('\n✅ Parsing complete.')

In [ ]:
# ── STEP 3: Clean & filter ────────────────────────────────────────────────
articles = clean_pipeline(all_articles)
print(f'✅ After cleaning: {len(articles)} articles remain')

In [ ]:
# ── STEP 4: Summarise & categorise ───────────────────────────────────────
for art in articles:
    art.summary  = summarise(art.body)
    art.takeaway = takeaway(art.body, art.title)
    art.category = categorise(art)
print('✅ Summarisation & categorisation complete.')

## 5️⃣  Generate Newsletter

In [ ]:
from collections import defaultdict
import csv

CAT_ORDER = ['Kenyan Banking News','Economic Updates','Regulatory & Policy Changes',
             'Fintech & Technology','Global Banking Trends','General Finance']

groups = defaultdict(list)
for a in articles: groups[a.category].append(a)

week_ending = datetime.now().strftime('%d %B %Y')
date_str    = datetime.now().strftime('%Y-%m-%d')

# ── Plain text ────────────────────────────────────────────────────────────
lines = ['='*65, '  HFC WEEKLY FINANCIAL INTELLIGENCE NEWSLETTER',
         f'  Week Ending: {week_ending}', '='*65, '']
for cat in CAT_ORDER:
    arts = groups.get(cat, [])
    if not arts: continue
    lines += ['-'*65, f'  {cat.upper()}', '-'*65, '']
    for a in arts:
        pub = a.pub_date.strftime('%d %b %Y') if a.pub_date else 'Date unknown'
        lines += [f'• {a.title}', f'  Source: {a.source}  |  {pub}',
                  f'  {a.summary}', f'  ➤ {a.takeaway}', f'  🔗 {a.url}', '']
txt_out = '\n'.join(lines)
txt_path = f'{OUTPUT_DIR}/hfc_newsletter_{date_str}.txt'
with open(txt_path, 'w') as f: f.write(txt_out)
print(f'✅ TXT saved: {txt_path}')

# ── CSV ───────────────────────────────────────────────────────────────────
csv_path = f'{OUTPUT_DIR}/hfc_newsletter_{date_str}.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['title','source','pub_date','category','url','summary','takeaway'])
    w.writeheader()
    for a in articles:
        w.writerow({'title':a.title,'source':a.source,
                    'pub_date':a.pub_date.strftime('%Y-%m-%d') if a.pub_date else '',
                    'category':a.category,'url':a.url,'summary':a.summary,'takeaway':a.takeaway})
print(f'✅ CSV saved: {csv_path}')

print(f'\n🎉 Newsletter complete! {len(articles)} articles processed.')

In [ ]:
# Preview the newsletter in the notebook
print(txt_out[:3000], '\n[…truncated]')

In [ ]:
# Download files from Colab
from google.colab import files
files.download(txt_path)
files.download(csv_path)

## 6️⃣  Automation Options

### Option A — GitHub Actions (recommended)
```yaml
# .github/workflows/newsletter.yml
name: Weekly Newsletter
on:
  schedule:
    - cron: '0 6 * * 1'   # Every Monday at 06:00 UTC
  workflow_dispatch:        # Allow manual trigger
jobs:
  generate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - run: python main.py --days 7 --output output/
      - uses: actions/upload-artifact@v4
        with: {name: newsletter, path: output/}
```

### Option B — Linux/Mac cron job
```bash
# Run every Monday at 07:00
0 7 * * 1 cd /path/to/hfc_newsletter && python main.py >> pipeline.log 2>&1
```

### Option C — Windows Task Scheduler
```
schtasks /create /tn 'HFC Newsletter' /tr 'python C:\hfc_newsletter\main.py' /sc weekly /d MON /st 07:00
```